# Project 1, CPSC 585
## Authors: Brian Chu (brianmchu42@csu.fullerton.edu), Jannik Hofmann (jhof@csu.fullerton.edu), Gabriel Villaruel ()

### 1. Clustering (70 points)

#### (a) Data preparation and preprocessing (5 points)
Standardize (z-score scaling) the four chemical properties and save them in a file named
“StdGasProperties.csv”, which will be used for the remaining tasks. Briefly explain:
* The importance of data standardization
* Feature mean and standard deviation before and after standardization

Clustering must be performed using only the four chemical features (excluding idx).

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

df = pd.read_csv("GasProperties.csv")

# skip Idx and only scale the four chemical properties
props = ['T', 'P', 'TC', 'SV']
idx = ['Idx']

preprocessor = ColumnTransformer(
    transformers = [
        ('scaler', StandardScaler(), props),
    ],
    remainder = 'passthrough'
)

transformed = preprocessor.fit_transform(df)

transformed_df = pd.DataFrame(transformed, columns = props + idx)

print('### ORIGINAL ###')
print(df)

print('### MEAN ###')
print(df.mean())
print('### STD ###')
print(df.std())

print('### TRANSFORMED ###')
print(transformed_df)

print('### MEAN ### ')
print(transformed_df.mean())
print('### STD ###')
print(transformed_df.std())

transformed_df.to_csv('StdGasProperties.csv', index=False)

### ORIGINAL ###
             T      P        TC          SV        Idx
0       253.15   17.7  0.026827  389.355242  54.102967
1       253.15   17.7  0.026970  388.007160  54.162740
2       253.15   17.7  0.026912  385.677846  54.232917
3       253.15   17.7  0.026815  384.292031  54.304434
4       253.15   17.7  0.026731  392.060606  54.070973
...        ...    ...       ...         ...        ...
419995  353.15  300.0  0.041824  435.341910  53.755453
419996  353.15  300.0  0.041518  426.840370  54.234870
419997  353.15  300.0  0.041744  428.069807  54.106685
419998  353.15  300.0  0.041607  430.308699  53.970731
419999  353.15  300.0  0.041669  432.137751  54.077029

[420000 rows x 5 columns]
### MEAN ###
T      303.150000
P      121.100000
TC       0.034696
SV     415.243807
Idx     53.972392
dtype: float64
### STD ###
T      34.156543
P      95.286764
TC      0.004071
SV     22.532554
Idx     0.251821
dtype: float64
### TRANSFORMED ###
              T         P        TC        SV 

Data standardization is important to regularize the data - numbers that are large in magnitude can be harder to process with some models. For example, since the magnitude of the `T` and `SV` columns in the data are more than two orders of magnitude larger than the `P` and `TC` columns, the `T` and `SV` columns can have an outsize effect on model predictions. Regularizing ensures that all relevant features are considered equally by the model.

As seen in the printout of the data, the original data has some particular mean and standard deviation, but when regularized, the mean becomes 0 and the standard deviation 1 (mathematically, with some floating point rounding error in practice)

#### (b) K-means clustering (10 points)
Perform K-Means with K=3 using and report:
* Initialization method used and initial centroid for each cluster
* Number of iterations until convergence
* Final centroids
* Cluster variances (within-cluster sum of squares)
* Number of samples assigned to each cluster

In [2]:
from sklearn.cluster import KMeans
from collections import Counter

cluster = KMeans(n_clusters=3, random_state=42)
cluster.fit(transformed_df[props]) # ignore Idx for now

print('Iterations before convergence: ', cluster.n_iter_)
print('Final centroids: ', cluster.cluster_centers_)

# calculate variances
transform = cluster.transform(transformed_df[props]) # gets distances to cluster


print('Samples per cluster: ', Counter(cluster.labels_))


Iterations before convergence:  7
Final centroids:  [[ 0.91934155 -0.11150019  0.81349347  0.89806952]
 [-0.63491456  1.20723778 -0.09903177 -0.96166327]
 [-0.92008584 -0.63841411 -1.12124672 -0.67222288]]
Samples per cluster:  Counter({np.int32(0): 196274, np.int32(2): 134644, np.int32(1): 89082})


In [3]:
print(cluster.transform(transformed_df[props]))

[[4.28497452 3.05624443 1.17534449]
 [4.29160215 3.03957851 1.17740159]
 [4.3522778  3.05822844 1.2371087 ]
 ...
 [2.28113745 3.24788637 4.6575407 ]
 [2.25549408 3.27725269 4.66452099]
 [2.25478314 3.32673928 4.69777013]]


#### (c) EM clustering (15 points)
Fit a Gaussian Mixture Model (GMM) with K=3 for clustering and report:
* Initialization method and initial parameters
* Convergence criterion (tolerance or max iterations)
* Covariance type (full, diagonal, spherical) with justification
* Mixture weights (𝜋_𝑘), means (𝜇_𝑘), covariances (Σ_𝑘) for each cluster
* For 3 samples, show probabilities 𝑝(𝑧 = 𝑘|𝑥)

In [4]:
from sklearn.mixture import GaussianMixture
import numpy as np

# initiate
em = GaussianMixture(
    n_components=3,
    init_params='kmeans',
    tol=1e-3,
    max_iter=100,
    covariance_type='full',
    random_state=42
)

# train
em.fit(transformed_df[props])

# print results
print("Converged:", em.converged_)
print("Iterations:", em.n_iter_)
print("Log Likelihood after Convergence:", em.lower_bound_)

print("Weights:", em.weights_)
print("Means:",em.means_)
print("Covariances:", em.covariances_)

sample_probs = em.predict_proba(transformed_df[props].iloc[:3])
print("Sample Probabilities:", np.round(sample_probs, 3))

Converged: True
Iterations: 22
Log Likelihood after Convergence: -1.1042902629326241
Weights: [0.37494468 0.38871665 0.23633867]
Means: [[ 1.02910681 -0.18180898  0.91829939  1.02764859]
 [-0.61887752  0.74748725 -0.27402545 -0.80043724]
 [-0.61475394 -0.94099044 -1.0061545  -0.31382122]]
Covariances: [[[ 0.19232868  0.06209208  0.22937179  0.14216409]
  [ 0.06209208  0.72126895  0.17829872 -0.17642661]
  [ 0.22937179  0.17829872  0.29269885  0.14493227]
  [ 0.14216409 -0.17642661  0.14493227  0.20417858]]

 [[ 0.52780954  0.222633    0.42842738  0.42449867]
  [ 0.222633    0.73686222  0.45764376 -0.03057279]
  [ 0.42842738  0.45764376  0.55070818  0.24300361]
  [ 0.42449867 -0.03057279  0.24300361  0.42664514]]

 [[ 0.3699395   0.0145347   0.37622812  0.35964249]
  [ 0.0145347   0.01811479  0.01951818  0.01186979]
  [ 0.37622812  0.01951818  0.38740695  0.37080814]
  [ 0.35964249  0.01186979  0.37080814  0.37789153]]]
Sample Probabilities: [[0.    0.003 0.997]
 [0.    0.002 0.998]
 [0

#### Report

##### Initialization Method
The Gaussian Mixture Model was initialized using the kmeans method for the initialization. The model uses 3 Gaussian components. Kmeans clsutering is first applied to the data to obtain the first parameter estimates for the misxture model. These estimates are then used as starting values for the EM algorithm. To ensure reproducibility of the cluster results random_state is set to 42.

##### Convergence Criterion
The EM algorithm was configured to run a maximum of 100 iterations. After 100 iterations the algorithm automatically stops to run to limit run time. The convergence tolerance is set to 10^-3, defining the minimum log-likelihood improvement between iterations. Once the improvement falls below this treshold the algorithm stops and the model is considered converged.The model converged after 22 iterations.

##### Covariance Type
The covariance type was set to 'full'. This allows each cluster to have its own full covariance matrix, meaning that correlations between features are considered. This allows the highest flexibility of cluster shape representation compared to diagonal or spherical covariance structures.

##### Estimated Cluster Parameters
After fitting the model, the following parameters where obtained:

Mixture Weights:
| Cluster | Weight |
|:---:|:---:|
| 1 | 0.37494468 |
| 2 | 0.38871665 |
| 3 | 0.23633867 |

The mixture weights represent the proportion of data points assigned to each cluster.

---

Cluster Means:
| Cluster | T | P | TC | SV |
|:---:|:---:|:---:|:---:|:---:|
| 1 | 1.02910681 | -0.18180898 | 0.91829939 | 1.02764859 |
| 2 | -0.61887752 | 0.74748725 | -0.27402545 | -0.80043724 |
| 3 | -0.61475394 | -0.94099044 | -1.0061545 | -0.31382122 |

The mean vectors represent the centers of the clusters in the standardized feature space. The numbers are between about -1 and +1 becuase the data was standardized using StandardScaler, which produces means at about 0 and standard deviations of about 1. Therefore, cluster centerrs are expressed in standard deviations from the global mean.

---

Covariance Matrices:

Covariance Matrix – Cluster 1

| Feature | T | P | TC | SV |
|:---:|:---:|:---:|:---:|:---:|
| T | 0.19232868 | 0.06209208 | 0.22937179 | 0.14216409 |
| P | 0.06209208 | 0.72126895 | 0.17829872 | -0.17642661 |
| TC | 0.22937179 | 0.17829872 | 0.29269885 | 0.14493227 |
| SV | 0.14216409 | -0.17642661 | 0.14493227 | 0.20417858 |

Covariance Matrix – Cluster 2

| Feature | T | P | TC | SV |
|:---:|:---:|:---:|:---:|:---:|
| T | 0.52780954 | 0.22263300 | 0.42842738 | 0.42449867 |
| P | 0.22263300 | 0.73686222 | 0.45764376 | -0.03057279 |
| TC | 0.42842738 | 0.45764376 | 0.55070818 | 0.24300361 |
| SV | 0.42449867 | -0.03057279 | 0.24300361 | 0.42664514 |

Covariance Matrix – Cluster 3

| Feature | T | P | TC | SV |
|:---:|:---:|:---:|:---:|:---:|
| T | 0.36993950 | 0.01453470 | 0.37622812 | 0.35964249 |
| P | 0.01453470 | 0.01811479 | 0.01951818 | 0.01186979 |
| TC | 0.37622812 | 0.01951818 | 0.38740695 | 0.37080814 |
| SV | 0.35964249 | 0.01186979 | 0.37080814 | 0.37789153 |

The covariance matrices describe the spread and orientation of each cluster in feature space. Larger variances indicate greater dispersion along a feature. Each Gaussian component has a 4x4 covariance matrix because the the clustering is performed in a 4 dimensional feature space.

##### Posterior Probabilities for Example Samples
To show the soft assignment properties, the posterior probablities were computed for 3 sample observations using predict_proba(). 

| Sample | Cluster 1 | Cluster 2 | Cluster 3 |
|:---:|:---:|:---:|:---:|
| 1 | 0 | 0.003 | 0.997 |
| 2 | 0 | 0.002 | 0.998 |
| 3 | 0 | 0 | 1 |

These probabilities represent the likelihood that the given observation belongs to each cluster. Unlike k-means, GMM provides soft assignments, meaning that each sample can belong to multiple clusters with different probabilities. These values indicate that the 3 samples belong strongly to the 3rd cluster.

#### (d) SOM clustering (20 points)
Train a 2D SOM and report:
* Grid size (e.g., 15x15, 20x20) with your justification
* Neighborhood function
* Learning rate schedule
* Number of iterations
* Similarity metric used

Extract clusters from SOM by clustering prototypes using K-Means: Since SOM does not directly
produce clusters, you must:
* Extract neuron weight vectors (prototypes)
* Apply K-Means with K=3 on neuron prototypes
* Assign each data point to the cluster of its BMU and report final cluster centroids and cluster sizes

In [5]:
from minisom import MiniSom
from sklearn.cluster import KMeans
from collections import Counter
import numpy as np

som = MiniSom(60, 60, 4,
              # interestingly enough the Mexican hat distribution makes some values explode to infinity
              neighborhood_function="gaussian",
              # no rhyme or reason for selection of values here, just vibes mostly lol
              learning_rate=0.75,
              decay_function="linear_decay_to_zero")

data = transformed_df[props].to_numpy()
som.train(data, num_iteration=500)

# Cluster SOM prototypes with K-Means
weights = som.get_weights()                        # shape: (60, 60, 4)
flat_weights = weights.reshape(3600, 4)            # shape: (3600, 4)

somCluster = KMeans(n_clusters=3, random_state=42)
somCluster.fit(flat_weights)

# somCluster.labels_ → cluster ID for each of the 3600 neurons
neuron_cluster_map = somCluster.labels_.reshape(60, 60)  # shape: (60, 60)

# Assign each data point via its BMU
point_labels = np.array([
    neuron_cluster_map[som.winner(x)]   # winner() returns (row, col) of BMU
    for x in data
])

print("Final centroids:\n", somCluster.cluster_centers_)
print("Cluster sizes:\n", Counter(point_labels))

Final centroids:
 [[-0.08588551 -0.16779775  0.27642749  0.41455643]
 [-0.19019835 -0.41414804 -0.40277258 -0.32636061]
 [ 0.17557946  0.46452765 -0.00395545 -0.17533381]]
Cluster sizes:
 Counter({np.int32(0): 162315, np.int32(1): 147622, np.int32(2): 110063})


*Grid size*: general heuristic for SOM models is that for N many samples, you want to have 5*sqrt(N) elements in the grid
For 420,000 samples, that's 3240 elements which is approximately grid dimension 60 x 60

*Neigborhood function* is Gaussian (the default, but experimenting with the Mexican hat yielded interesting results as some weights exploded to infinity

*Learning rate* schedule is starting at 0.75 with a linear decay to zero. Values chosen arbitrarily.

*Number of iterations* is 500, again arbitrarily.

Similarity metric defaults to Euclidean, which maps well to clustering methods on a SOM.

#### (e) Cluster evaluation and interpretation (20 points)

##### Silhouette score

##### Wobbe index

Wobbe indices are used to compare gases for combustion and are derived by dividing the heating value by the square root of the specific gravity. Gases with similar Wobbe indices are considered interchangeable when designing combustion systems. For the purposes of classification, we'll divide the Wobbe indices into three parts (bottom 33% = "regular", middle = "medium", top = "premium")

##### Adjusted rand index

### 2. Classification (50 points)

Split the dataset into training (70%), validation set (15%), and testing set (15%) using a fixed random seed.
All final performance metrics must be reported on the test set.

In [6]:
from sklearn.model_selection import train_test_split

# relabel Idx to Wobbe classification as mentioned above
minIdx, maxIdx = min(transformed_df['Idx']), max(transformed_df['Idx'])
threshold = (maxIdx - minIdx) / 3

def getGrade(idx):
    if idx < minIdx + threshold:
        return 1
    if idx < minIdx + 2 * threshold:
        return 2
    return 3

transformed_df['Idx'] = transformed_df['Idx'].map(getGrade)

# split data into train, val, test
training, vt = train_test_split(transformed_df, test_size=0.3, random_state=42)
validation, testing = train_test_split(vt, test_size=0.5, random_state=42)

#### (a) MLP Classifier (20 points)
Train a MLP to predict quality levels (Regular, Medium, Premium) and report:
* Topology (layers and number of neurons)
* Activation functions
* Optimizer (gradient method)
* Learning rate
* Number of epochs
* Training methods and batch size
* Regularization method if any

Evaluate on test set using confusion matrix, % accuracy, and F1-score.